# Conexion a drive y librerias

In [ ]:
from google.colab import drive     # Conectar a Google drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import xarray as xr
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
!pip install cftime

# Abriendo archivo de estaciones

In [ ]:
ruta_leyenda = "/content/drive/MyDrive/4Students_2026/Data/Estaciones_Meteorológicas_Peru.xlsx" #verificar si esta es tu ruta
estaciones_leyenda = pd.read_excel(ruta_leyenda, header=None)
estaciones_leyenda.head()

,0,1,2,3,4,5,6,7,8
0,ho00000130,-3,48,46.8,-80,27,28.8,113,RICAPLAYA
1,ho00000132,-3,30,28.8,-80,27,25.2,7,PUERTOPIZARRO
2,ho00000134,-3,34,33.6,-80,14,13.2,45,PAPAYAL
3,ho00000135,-3,26,27.6,-80,19,19.2,6,ELSALTO
4,ho00000172,-4,0,10.8,-73,9,39.6,94,TAMSHIYACU


In [ ]:
estaciones_leyenda.rename(columns={0: "Codigo",
                                   1: "Latitud (grados)",
                                   2: "Latitud (minutos)",
                                   3: "Latitud (segundos)",
                                   4: "Longitud (grados)",
                                   5: "Longitud (minutos)",
                                   6: "Longitud (segundos)",
                                   7: "Altitud (msnm)",
                                   8: "Nombre"}, inplace=True)
estaciones_leyenda["Latitud"] = estaciones_leyenda["Latitud (grados)"] - estaciones_leyenda["Latitud (minutos)"]/60 - estaciones_leyenda["Latitud (segundos)"]/3600
estaciones_leyenda["Longitud"] = estaciones_leyenda["Longitud (grados)"] - estaciones_leyenda["Longitud (minutos)"]/60 - estaciones_leyenda["Longitud (segundos)"]/3600

estaciones_leyenda.head()

,Codigo,Latitud (grados),Latitud (minutos),Latitud (segundos),Longitud (grados),Longitud (minutos),Longitud (segundos),Altitud (msnm),Nombre,Latitud,Longitud
0,ho00000130,-3,48,46.8,-80,27,28.8,113,RICAPLAYA,-3.813,-80.458
1,ho00000132,-3,30,28.8,-80,27,25.2,7,PUERTOPIZARRO,-3.508,-80.457
2,ho00000134,-3,34,33.6,-80,14,13.2,45,PAPAYAL,-3.576,-80.237
3,ho00000135,-3,26,27.6,-80,19,19.2,6,ELSALTO,-3.441,-80.322
4,ho00000172,-4,0,10.8,-73,9,39.6,94,TAMSHIYACU,-4.003,-73.161


In [ ]:
# IMPORTANTEEEE
#Aqui ustedes deben poner su archivo de datos completos, por favor tenganlo en su carpeta
df_region = pd.read_excel("/content/drive/MyDrive/4Students_2026/Data/Selva/Datos_completos/datos_completos_Selva.xlsx")
df_region.head()

,Fecha,tmax_obs,tmin_obs,pp_obs,tmax,tmin,pp,Tmax_origen,Tmax_completo,Tmin_origen,Tmin_completo,Pp_origen,Pp_completo,Codigo,Nombre,Latitud,Longitud,Altitud (msnm)
0,1965-01-01,30.5,NaN,0.0,NaN,NaN,NaN,obs,30.5,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
1,1965-01-02,29.6,NaN,0.0,NaN,NaN,NaN,obs,29.6,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
2,1965-01-03,27.8,NaN,0.0,NaN,NaN,NaN,obs,27.8,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
3,1965-01-04,30.9,NaN,0.0,NaN,NaN,NaN,obs,30.9,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
4,1965-01-05,31.0,NaN,0.0,NaN,NaN,NaN,obs,31.0,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645


In [ ]:
df_region["Codigo"].unique()

array(['ho00000833', 'ho00000840', 'ho00000837', 'ho00000830',
       'ho00000846'], dtype=object)

# Funciones

In [ ]:
def extraer_serie_base(ruta_carpeta, lat_est, lon_est, variable, escenario):
    """
    Función interna que extrae una serie de tiempo simple [Fecha, Valor].
    INCLUYE SOLUCIÓN PARA CALENDARIOS CMIP6 Y ACTUALIZACIÓN DE XARRAY.
    """
    mapa_prefix = {'pr': 'pr_day', 'tmax': 'tasmax_day', 'tmin': 'tasmin_day'}
    prefix = mapa_prefix.get(variable.lower())

    patron = os.path.join(ruta_carpeta, f"{prefix}_*_{escenario}_*.nc")
    archivos = glob.glob(patron)

    if not archivos:
        return None

    try:
        # --- SOLUCIÓN AL WARNING DE XARRAY ---
        try:
            # Sintaxis para versiones nuevas de xarray (la que pide el warning)
            time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
            ds = xr.open_dataset(archivos[0], decode_times=time_coder)
        except AttributeError:
            # Sintaxis para versiones clásicas de xarray
            ds = xr.open_dataset(archivos[0], use_cftime=True)
        # -------------------------------------

        var_interna = prefix.split('_')[0]
        lon_modelo = lon_est if ds.lon.max() <= 180 else (lon_est + 360 if lon_est < 0 else lon_est)

        punto = ds[var_interna].sel(lat=lat_est, lon=lon_modelo, method='nearest')
        df_res = punto.to_dataframe().reset_index()

        # Conversión de unidades
        if var_interna == 'pr':
            df_res[var_interna] = df_res[var_interna] * 86400 # kg/m2/s a mm/dia
        elif var_interna in ['tasmax', 'tasmin']:
            df_res[var_interna] = df_res[var_interna] - 273.15 # K a °C

        # --- LA MAGIA DEL CALENDARIO ---
        if 'time' in df_res.columns:
            # Extraemos las fechas crudas del modelo a texto
            fechas_str = df_res['time'].astype(str).str.slice(0, 10)

            # Dejamos que Pandas filtre:
            # Guarda los días reales y pone NaT a los falsos (ej. 30 feb)
            df_res['time'] = pd.to_datetime(fechas_str, errors='coerce')

            # Eliminamos los días falsos inventados por el modelo
            df_res = df_res.dropna(subset=['time'])

        # Estandarizamos
        df_res = df_res[['time', var_interna]].rename(columns={'time': 'Fecha', var_interna: 'Valor'})
        ds.close()

        return df_res

    except Exception as e:
        print(f"Error interno en {variable} {escenario}: {e}")
        return None

In [ ]:
def unir_estacion_cmip6(ruta_carpeta, df_estaciones, codigo_estacion, nombre_modelo, df_observaciones, ruta_salida="."):
    """
    Función maestra: Empalma historical con SSPs para PR, TMAX y TMIN,
    agrega los datos observados de la estación, y guarda un CSV consolidado.
    """
    print(f"\n--- Procesando Estación: {codigo_estacion} | Modelo: {nombre_modelo} ---")

    estacion = df_estaciones[df_estaciones['Codigo'] == codigo_estacion]
    if estacion.empty:
        print(f"❌ El código {codigo_estacion} no existe en metadatos.")
        return None

    lat_est = estacion['Latitud'].values[0]
    lon_est = estacion['Longitud'].values[0]

    variables = ['pr', 'tmax', 'tmin']
    escenarios_futuros = ['ssp245', 'ssp585']

    df_consolidado = pd.DataFrame()

    # 1. Extraer datos del modelo CMIP6
    for var in variables:
        df_hist = extraer_serie_base(ruta_carpeta, lat_est, lon_est, var, 'historical')

        for ssp in escenarios_futuros:
            df_futuro = extraer_serie_base(ruta_carpeta, lat_est, lon_est, var, ssp)

            if df_hist is not None and df_futuro is not None:
                df_empalmado = pd.concat([df_hist, df_futuro])
                df_empalmado = df_empalmado.drop_duplicates(subset=['Fecha'], keep='last').sort_values('Fecha')

                col_name = f"{var.upper()}_{ssp}"
                df_empalmado = df_empalmado.rename(columns={'Valor': col_name})

                if df_consolidado.empty:
                    df_consolidado = df_empalmado
                else:
                    df_consolidado = pd.merge(df_consolidado, df_empalmado, on='Fecha', how='outer')
            else:
                print(f"⚠️ Faltan datos para armar la serie {var.upper()} {ssp}")

    if not df_consolidado.empty:

        # ================================================================
        # NUEVO: AGREGAR LOS DATOS OBSERVADOS HISTÓRICOS (TÚ EXCEL)
        # ================================================================
        print("📊 Agregando datos observados (históricos)...")

        # Filtramos tu df_region para quedarnos solo con la estación actual
        df_obs_estacion = df_observaciones[df_observaciones['Codigo'] == codigo_estacion].copy()

        # Seleccionamos las columnas útiles y las renombramos
        columnas_utiles = ['Fecha', 'Tmax_completo', 'Tmin_completo', 'Pp_completo']
        df_obs_estacion = df_obs_estacion[columnas_utiles].rename(columns={
            'Tmax_completo': 'Tmax_obs',
            'Tmin_completo': 'Tmin_obs',
            'Pp_completo': 'Pp_obs'
        })

        # Aseguramos que la columna 'Fecha' sea del mismo tipo en ambos lados para evitar errores de cruce
        df_consolidado['Fecha'] = pd.to_datetime(df_consolidado['Fecha'])
        df_obs_estacion['Fecha'] = pd.to_datetime(df_obs_estacion['Fecha'])

        # Unimos todo (outer join para no perder ni un día del modelo ni de las observaciones)
        df_final = pd.merge(df_consolidado, df_obs_estacion, on='Fecha', how='outer')
        # ================================================================

        # Ordenamos las columnas para que quede ordenado cronológicamente
        df_final = df_final.sort_values('Fecha').reset_index(drop=True)

        # Guardar en CSV
        nombre_archivo = f"comparacion_{nombre_modelo}_{codigo_estacion}.csv"
        ruta_completa = os.path.join(ruta_salida, nombre_archivo)
        df_final.to_csv(ruta_completa, index=False)

        print(f"✅ ¡Éxito! Archivo guardado: {nombre_archivo}")
        return df_final
    else:
        print("❌ No se pudo construir el DataFrame consolidado.")
        return None

# Aplicando funciones

In [ ]:
df_region["Codigo"].unique()

array(['ho00000833', 'ho00000840', 'ho00000837', 'ho00000830',
       'ho00000846'], dtype=object)

In [ ]:
codigos = df_region["Codigo"].unique() #copio los codigos de arriba
codigos

array(['ho00000833', 'ho00000840', 'ho00000837', 'ho00000830',
       'ho00000846'], dtype=object)

In [ ]:
ruta_modelo_access = "/content/drive/MyDrive/4Students_2026/Data/Costa_Norte/ACCESS-CM2"
#MODELOS QUE ANTES SALIAN ERRORES
ruta_modelo_kace = "/content/drive/MyDrive/4Students_2026/Data/Sierra_Norte/KACE-1-0-G"
ruta_noresm2 = "/content/drive/MyDrive/4Students_2026/Data/Sierra_Norte/NorESM2-MM"
ruta_bcc_csm2 = "/content/drive/MyDrive/4Students_2026/Data/Sierra_Sur/BCC-CSM2-MR"
#CARPETA GUARDAR
ruta_guardado = "/content/drive/MyDrive/4Students_2026/Data/Selva/Resultados_CMIP6" # Cambia esto a tu carpeta deseada

In [ ]:
codigos

array(['ho00000833', 'ho00000840', 'ho00000837', 'ho00000830',
       'ho00000846'], dtype=object)

In [ ]:
ruta_CNRM_CM6 = "/content/drive/MyDrive/4Students_2026/Data/Costa_Centro/CNRM-CM6-1/CNRM-CM6-1"

In [ ]:
df_region

,Fecha,tmax_obs,tmin_obs,pp_obs,tmax,tmin,pp,Tmax_origen,Tmax_completo,Tmin_origen,Tmin_completo,Pp_origen,Pp_completo,Codigo,Nombre,Latitud,Longitud,Altitud (msnm)
0,1965-01-01,30.5,NaN,0.0,NaN,NaN,NaN,obs,30.5,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
1,1965-01-02,29.6,NaN,0.0,NaN,NaN,NaN,obs,29.6,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
2,1965-01-03,27.8,NaN,0.0,NaN,NaN,NaN,obs,27.8,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
3,1965-01-04,30.9,NaN,0.0,NaN,NaN,NaN,obs,30.9,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
4,1965-01-05,31.0,NaN,0.0,NaN,NaN,NaN,obs,31.0,sin_dato,NaN,obs,0.0,ho00000833,APLAO,-16.069,-72.491,645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100435,2019-12-27,24.6,18.8,NaN,NaN,NaN,NaN,obs,24.6,obs,18.8,sin_dato,NaN,ho00000846,PUNTACOLES,-17.699,-71.374,25
100436,2019-12-28,25.8,18.2,NaN,NaN,NaN,NaN,obs,25.8,obs,18.2,sin_dato,NaN,ho00000846,PUNTACOLES,-17.699,-71.374,25
100437,2019-12-29,25.2,18.4,NaN,NaN,NaN,NaN,obs,25.2,obs,18.4,sin_dato,NaN,ho00000846,PUNTACOLES,-17.699,-71.374,25
100438,2019-12-30,26.2,18.6,NaN,NaN,NaN,NaN,obs,26.2,obs,18.6,sin_dato,NaN,ho00000846,PUNTACOLES,-17.699,-71.374,25


In [ ]:
# todas las estaciones en bucle
for codigo in codigos:
    df_modelo5 = unir_estacion_cmip6(
        ruta_carpeta=ruta_CNRM_CM6, #RUTA MODELO
        df_estaciones=estaciones_leyenda,
        codigo_estacion=codigo,
        nombre_modelo="CNRM-CM6-1",
        df_observaciones=df_region,    # <--- AQUÍ LE PASAMOS TU EXCEL DE LA SELVA
        ruta_salida=ruta_guardado
    )

# Revisar el resultado final
df_modelo5.head()


--- Procesando Estación: ho00000833 | Modelo: CNRM-CM6-1 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_CNRM-CM6-1_ho00000833.csv

--- Procesando Estación: ho00000840 | Modelo: CNRM-CM6-1 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_CNRM-CM6-1_ho00000840.csv

--- Procesando Estación: ho00000837 | Modelo: CNRM-CM6-1 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_CNRM-CM6-1_ho00000837.csv

--- Procesando Estación: ho00000830 | Modelo: CNRM-CM6-1 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_CNRM-CM6-1_ho00000830.csv

--- Procesando Estación: ho00000846 | Modelo: CNRM-CM6-1 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_CNRM-CM6-1_ho00000846.csv


,Fecha,PR_ssp245,PR_ssp585,TMAX_ssp245,TMAX_ssp585,TMIN_ssp245,TMIN_ssp585,Tmax_obs,Tmin_obs,Pp_obs
0,1965-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1965-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1965-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1965-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1965-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# todas las estaciones en bucle
for codigo in codigos:
    df_modelo1 = unir_estacion_cmip6(
        ruta_carpeta=ruta_modelo_kace, #RUTA MODELO
        df_estaciones=estaciones_leyenda,
        codigo_estacion=codigo,
        nombre_modelo="KACE-1-0-G",
        df_observaciones=df_region,    # <--- AQUÍ LE PASAMOS TU EXCEL DE LA SELVA
        ruta_salida=ruta_guardado
    )

# Revisar el resultado final
df_modelo1.head()


--- Procesando Estación: ho00000833 | Modelo: KACE-1-0-G ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_KACE-1-0-G_ho00000833.csv

--- Procesando Estación: ho00000840 | Modelo: KACE-1-0-G ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_KACE-1-0-G_ho00000840.csv

--- Procesando Estación: ho00000837 | Modelo: KACE-1-0-G ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_KACE-1-0-G_ho00000837.csv

--- Procesando Estación: ho00000830 | Modelo: KACE-1-0-G ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_KACE-1-0-G_ho00000830.csv

--- Procesando Estación: ho00000846 | Modelo: KACE-1-0-G ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_KACE-1-0-G_ho00000846.csv


,Fecha,PR_ssp245,PR_ssp585,TMAX_ssp245,TMAX_ssp585,TMIN_ssp245,TMIN_ssp585,Tmax_obs,Tmin_obs,Pp_obs
0,1965-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1965-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1965-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1965-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1965-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# todas las estaciones en bucle
for codigo in codigos:
    df_modelo2 = unir_estacion_cmip6(
        ruta_carpeta=ruta_noresm2,#cambias tu carpeta donde estan los datos del modelo
        df_estaciones=estaciones_leyenda,
        codigo_estacion=codigo,
        nombre_modelo="NorESM2-MM",
        df_observaciones=df_region,    # <--- AQUÍ LE PASAMOS TU EXCEL DE LA SELVA
        ruta_salida=ruta_guardado
    )

# Revisar el resultado final
df_modelo2.head()


--- Procesando Estación: ho00000833 | Modelo: NorESM2-MM ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_NorESM2-MM_ho00000833.csv

--- Procesando Estación: ho00000840 | Modelo: NorESM2-MM ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_NorESM2-MM_ho00000840.csv

--- Procesando Estación: ho00000837 | Modelo: NorESM2-MM ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_NorESM2-MM_ho00000837.csv

--- Procesando Estación: ho00000830 | Modelo: NorESM2-MM ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_NorESM2-MM_ho00000830.csv

--- Procesando Estación: ho00000846 | Modelo: NorESM2-MM ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_NorESM2-MM_ho00000846.csv


,Fecha,PR_ssp245,PR_ssp585,TMAX_ssp245,TMAX_ssp585,TMIN_ssp245,TMIN_ssp585,Tmax_obs,Tmin_obs,Pp_obs
0,1965-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1965-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1965-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1965-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1965-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# todas las estaciones en bucle
for codigo in codigos:
    df_modelo3 = unir_estacion_cmip6(
        ruta_carpeta=ruta_modelo_access,#cambias tu carpeta donde estan los datos del modelo
        df_estaciones=estaciones_leyenda,
        codigo_estacion=codigo,
        nombre_modelo="ACCESS-CM2",
        df_observaciones=df_region,    # <--- AQUÍ LE PASAMOS TU EXCEL DE LA SELVA
        ruta_salida=ruta_guardado
    )

# Revisar el resultado final
df_modelo3.head()


--- Procesando Estación: ho00000833 | Modelo: ACCESS-CM2 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_ACCESS-CM2_ho00000833.csv

--- Procesando Estación: ho00000840 | Modelo: ACCESS-CM2 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_ACCESS-CM2_ho00000840.csv

--- Procesando Estación: ho00000837 | Modelo: ACCESS-CM2 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_ACCESS-CM2_ho00000837.csv

--- Procesando Estación: ho00000830 | Modelo: ACCESS-CM2 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_ACCESS-CM2_ho00000830.csv

--- Procesando Estación: ho00000846 | Modelo: ACCESS-CM2 ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_ACCESS-CM2_ho00000846.csv


,Fecha,PR_ssp245,PR_ssp585,TMAX_ssp245,TMAX_ssp585,TMIN_ssp245,TMIN_ssp585,Tmax_obs,Tmin_obs,Pp_obs
0,1965-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1965-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1965-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1965-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1965-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# todas las estaciones en bucle
for codigo in codigos:
    df_modelo3 = unir_estacion_cmip6(
        ruta_carpeta=ruta_bcc_csm2,#cambias tu carpeta donde estan los datos del modelo
        df_estaciones=estaciones_leyenda,
        codigo_estacion=codigo,
        nombre_modelo="BCC-CSM2-MR",
        df_observaciones=df_region,    # <--- AQUÍ LE PASAMOS TU EXCEL DE LA SELVA
        ruta_salida=ruta_guardado
    )

# Revisar el resultado final
df_modelo3.head()


--- Procesando Estación: ho00000833 | Modelo: BCC-CSM2-MR ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_BCC-CSM2-MR_ho00000833.csv

--- Procesando Estación: ho00000840 | Modelo: BCC-CSM2-MR ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_BCC-CSM2-MR_ho00000840.csv

--- Procesando Estación: ho00000837 | Modelo: BCC-CSM2-MR ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_BCC-CSM2-MR_ho00000837.csv

--- Procesando Estación: ho00000830 | Modelo: BCC-CSM2-MR ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_BCC-CSM2-MR_ho00000830.csv

--- Procesando Estación: ho00000846 | Modelo: BCC-CSM2-MR ---
📊 Agregando datos observados (históricos)...
✅ ¡Éxito! Archivo guardado: comparacion_BCC-CSM2-MR_ho00000846.csv


,Fecha,PR_ssp245,PR_ssp585,TMAX_ssp245,TMAX_ssp585,TMIN_ssp245,TMIN_ssp585,Tmax_obs,Tmin_obs,Pp_obs
0,1965-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1965-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1965-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1965-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1965-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# TAREA DE UN SOLO MODELO Y DE UNA ESTACION
Corroborar si el código automatizado esta correcto